<a href="https://colab.research.google.com/github/ghirailghiro/AMD-Ghirardelli-Michele/blob/main/AMD_Michele_Ghirardelli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The outputs stored in this notebook come from a run with FULL_DATASET = True, i.e. on the whole 2020 dataset. The default value is False, which restricts the analysis to a single month in order to keep the execution time on Google Colab reasonable.

# `Init Enviroment`

In this section all the python dependecies and datasets that we are going to use inside the project

In [1]:
!pip install -q pyspark
!pip install -q kaggle
!pip install -q kagglehub[pandas-datasets,hf-datasets]

In [2]:
FULL_DATASET = False
EDGES_RANGES_ANALYSIS = True

In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
os.environ['KAGGLE_USERNAME'] = "xxxxxx"
os.environ['KAGGLE_KEY'] = "xxxxxx"

# Set the path to the file you'd like to load
file_path_articles = "nyt-articles-2020.csv"
file_path_comments = "nyt-comments-2020.csv"

# Load all the comments
df_comments = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "benjaminawd/new-york-times-articles-comments-2020",
  file_path_comments,
)


/tmp/ipykernel_2257/2091549453.py:9: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_comments = kagglehub.load_dataset(


100%|██████████| 994M/994M [00:13<00:00, 74.8MB/s]

Extracting zip of nyt-comments-2020.csv...



/usr/local/lib/python3.13/dist-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [4]:
if not FULL_DATASET:
  df_comments = df_comments[df_comments.createDate.str[:7].isin(["2020-01"])]

# Pre-Processing and Analysis of the dataset

In [5]:
# total comments
num_total_comments = df_comments.shape[0]

# comments in response to other comments
num_comments_with_parent_notna = df_comments[df_comments.parentID.notna()].shape[0]

print(f"total number of comments {num_total_comments}")
print(f"number of comments with parent {num_comments_with_parent_notna}")
print(f"percentage of comments with parents {num_comments_with_parent_notna/num_total_comments * 100} %")

# all unique users so all the unique nodes of the graph
all_number_of_users = df_comments.userID.unique()

# users that commented on other comment
not_parents_ids = df_comments[df_comments.parentID.notna()].userID.unique()
not_parents = len(not_parents_ids)

# root unique comments
comments_parents_ids = df_comments[df_comments.parentID.notna()].parentID.unique()
# users that have inserted a root comment
parents_ids = df_comments[df_comments.commentID.isin(comments_parents_ids)].userID.unique()

# users that inserted a root comment and also a response to another comment
intersection = set(not_parents_ids).intersection(parents_ids)
print("-----------------------------------------------------------------------")
print(f"All unique users {len(all_number_of_users)}")
print(f"Unique users that have one root comment {len(parents_ids)} is {len(parents_ids) / len(all_number_of_users) * 100} %")
print(f"Unique users that commented one time to another user {not_parents} is {not_parents / len(all_number_of_users) * 100} %")
print(f"Unique users that have one root comment and commented to other users {len(intersection)} is {len(intersection) / len(all_number_of_users) * 100} %")

total number of comments 4986461
number of comments with parent 2160907
percentage of comments with parents 43.33548382309618 %
-----------------------------------------------------------------------
All unique users 403025
Unique users that have one root comment 182514 is 45.28602444017121 %
Unique users that commented one time to another user 197853 is 49.091991811922334 %
Unique users that have one root comment and commented to other users 117572 is 29.172383847155885 %


Clean from orfans, self loop and anonymous users

In [6]:
# get all the comments
all_userID_commentID = df_comments[["commentID", "userID"]]

# merge all the possible comments with just childs comments
df_childs_comments = df_comments[df_comments.parentID.notna()][["commentID", "userID", "parentID"]]
df_child_parent_arc = df_childs_comments.merge(all_userID_commentID, how="left",left_on="parentID", right_on="commentID", suffixes=("_src", "_dst"))

df_child_parent_arc_len = df_child_parent_arc.shape[0]

# count orfans amount sometime moderation can cancel parent comments
orfans_amount = df_child_parent_arc[df_child_parent_arc.commentID_dst.isna()].shape[0]


df_child_parent_arc_cleaned = df_child_parent_arc[df_child_parent_arc.commentID_dst.notna()]

df_child_parent_arc_cleaned_no_dupl = df_child_parent_arc_cleaned[["userID_src", "userID_dst"]].drop_duplicates()

df_child_parent_arc_cleaned_no_dupl_len = df_child_parent_arc_cleaned_no_dupl.shape[0]

# self loop reponses, users that response to themselfs
cappio_value = df_child_parent_arc_cleaned_no_dupl[df_child_parent_arc_cleaned_no_dupl.userID_src == df_child_parent_arc_cleaned_no_dupl.userID_dst].shape[0]
df_child_parent_arc_cleaned_no_dupl = df_child_parent_arc_cleaned_no_dupl[df_child_parent_arc_cleaned_no_dupl.userID_src != df_child_parent_arc_cleaned_no_dupl.userID_dst]

# cleaning anonymous users
total_before_anon = df_child_parent_arc_cleaned_no_dupl.shape[0]

ids_anonymous = df_comments[df_comments.isAnonymous].userID.unique()

df_child_parent_arc_cleaned_final = df_child_parent_arc_cleaned_no_dupl[df_child_parent_arc_cleaned_no_dupl.userID_dst.isin(ids_anonymous) == False]
df_child_parent_arc_cleaned_final = df_child_parent_arc_cleaned_final[df_child_parent_arc_cleaned_final.userID_src.isin(ids_anonymous) == False]

df_anon = df_child_parent_arc_cleaned_no_dupl[(df_child_parent_arc_cleaned_no_dupl.userID_dst.isin(ids_anonymous) == True) | (df_child_parent_arc_cleaned_no_dupl.userID_src.isin(ids_anonymous) == True)]

df_anon_len = df_anon.shape[0]
print(f"Percentage of orfans cleaned {100 * (orfans_amount/df_child_parent_arc_len)} %")
print(f"Percentage of cappio comments cleaned {100 * (cappio_value/df_child_parent_arc_cleaned_no_dupl_len)}")
print(f"Percentage of anonymous users cleaned {100 * (df_anon_len/total_before_anon)} %")

Percentage of orfans cleaned 0.6516708030470539 %
Percentage of cappio comments cleaned 0.5240054994536447
Percentage of anonymous users cleaned 0.0 %


In [46]:
#total nodes and edges of the graph

userID_dst_set = set(list(df_child_parent_arc_cleaned_final.userID_dst.unique()))
userID_src_set = set(list(df_child_parent_arc_cleaned_final.userID_src.unique()))
intersection_cleaned = userID_dst_set.intersection(userID_src_set)

number_of_nodes = len(userID_dst_set.union(userID_src_set))
number_of_edges = df_child_parent_arc_cleaned_final.shape[0]



print(f"Number of nodes {number_of_nodes}")
print(f"Number of edges {number_of_edges}")
print(f"Number of edges per node {number_of_edges/number_of_nodes}")
print(f"Node with at least a response {len(userID_dst_set)}")
print(f"Node with at least a written response {len(userID_src_set)}")
print(f"Node with at least a written and a response {len(intersection_cleaned)}")
print(f"Checksum {len(userID_dst_set) + len(userID_src_set) - len(intersection_cleaned) == number_of_nodes}")


Number of nodes 262184
Number of edges 1967288
Number of edges per node 7.503463216672261
Node with at least a response 181962
Node with at least a written response 196093
Node with at least a written and a response 115871
Checksum True


In [56]:
import pandas as pd
pd.set_option('display.float_format', '{:,.2f}'.format)

rows_preprocessing = [
    {"step": "Total comments",
     "value": num_total_comments,
     "percentage": None},

    {"step": "Comments with parent (replies)",
     "value": num_comments_with_parent_notna,
     "percentage": 100 * num_comments_with_parent_notna / num_total_comments},

    {"step": "Orphans discarded",
     "value": orfans_amount,
     "percentage": 100 * orfans_amount / df_child_parent_arc_len},

    {"step": "Self-loops removed",
     "value": cappio_value,
     "percentage": 100 * cappio_value / df_child_parent_arc_cleaned_no_dupl_len},

    {"step": "Anonymous edges removed",
     "value": df_anon_len,
     "percentage": 100 * df_anon_len / total_before_anon},

]

df_preprocessing = pd.DataFrame(rows_preprocessing)
rows_graph = [
    {"step": "All users in the dataset",
     "value": len(all_number_of_users),
     "percentage": None},

    {"step": "Graph nodes",
     "value": number_of_nodes,
     "percentage": 100 * number_of_nodes / len(all_number_of_users)},

    {"step": "Final edges",
     "value": number_of_edges,
     "percentage": None},

    {"step": "Average out-degree",
     "value": number_of_edges / number_of_nodes,
     "percentage": None},

    {"step": "Users who replied (out-degree > 0)",
     "value": len(userID_src_set),
     "percentage": 100 * len(userID_src_set) / number_of_nodes},

    {"step": "Users who received replies (in-degree > 0)",
     "value": len(userID_dst_set),
     "percentage": 100 * len(userID_dst_set) / number_of_nodes},

    {"step": "Users in both sets",
     "value": len(intersection_cleaned),
     "percentage": 100 * len(intersection_cleaned) / number_of_nodes},
]

df_graph_structure = pd.DataFrame(rows_graph)



In [57]:
df_preprocessing

,step,value,percentage
0,Total comments,4986461,NaN
1,Comments with parent (replies),2160907,43.34
2,Orphans discarded,14082,0.65
3,Self-loops removed,10363,0.52
4,Anonymous edges removed,0,0.00


In [58]:
df_graph_structure

,step,value,percentage
0,All users in the dataset,"403,025.00",NaN
1,Graph nodes,"262,184.00",65.05
2,Final edges,"1,967,288.00",NaN
3,Average out-degree,7.50,NaN
4,Users who replied (out-degree > 0),"196,093.00",74.79
5,Users who received replies (in-degree > 0),"181,962.00",69.40
6,Users in both sets,"115,871.00",44.19


In [11]:
df_child_parent_arc_cleaned_final.to_csv("df_child_parent_arc_cleaned_final.csv", index=False)

# Numpy Implementation

In [16]:
import numpy as np
import pandas as pd

src_dsr = pd.read_csv("df_child_parent_arc_cleaned_final.csv", dtype={"userID_src": "int64", "userID_dst": "int64"})

src = src_dsr.userID_src.to_numpy()
dst = src_dsr.userID_dst.to_numpy()

users_converted_indexes = np.unique(np.concatenate([src , dst]))

src_indexs = np.searchsorted(users_converted_indexes, src)
dst_indexs = np.searchsorted(users_converted_indexes, dst)
n = len(users_converted_indexes)
Beta = 0.85


In [17]:
# sanity checks
assert len(src) == len(src_indexs)
assert len(dst) == len(dst_indexs)
assert np.all(users_converted_indexes[src_indexs] == src)

In [18]:
import time
def dense_implementation(k, src_indexs, dst_indexs):
    v = np.zeros(k) + 1/k
    mask = (src_indexs < k) & (dst_indexs < k)
    src_sub = src_indexs[mask]
    dst_sub = dst_indexs[mask]
    M = np.zeros((k, k))
    bin_src_indexs_sub = np.bincount(src_sub,  minlength=k)
    for index in range(len(src_sub)):
      j = src_sub[index]
      degree = bin_src_indexs_sub[j]
      i=dst_sub[index]
      M[i,j] = 1/degree
    count_iterations = 0
    for iter in range(50):
      count_iterations += 1
      print(f"Iteration number {iter}")
      v_new = np.zeros(k)
      v_old = v.copy()
      S = v[bin_src_indexs_sub == 0].sum()
      v_new = M @ v
      v = Beta * (v_new + S/k) + ((1-Beta)/k)
      print(f"Somma a 1 : {abs(v.sum() - 1) < 1e-10}")
      residuo = np.abs(v - v_old).sum()
      print(f"Residuo : {residuo}")
      if residuo < 1e-8:
        print(f"Residuo : {residuo}")
        break
    return v, count_iterations, M.nbytes, bin_src_indexs_sub.sum()

def sparse_implementation(k, src_indexs, dst_indexs):
    v = np.zeros(k) + 1/k
    mask = (src_indexs < k) & (dst_indexs < k)
    src_sub = src_indexs[mask]
    dst_sub = dst_indexs[mask]
    bin_src_indexs_sub = np.bincount(src_sub, minlength=k)
    count_iterations = 0
    for iter in range(50):
      count_iterations += 1
      print(f"Iteration number {iter}")
      v_new = np.zeros(k)
      v_old = v.copy()
      S = v[bin_src_indexs_sub == 0].sum()
      for index in range(len(src_sub)):
        j = src_sub[index]
        degree = bin_src_indexs_sub[j]
        contribute = v[j] * 1/degree
        v_new[dst_sub[index]] += contribute
      v = Beta * (v_new + S/k) + ((1-Beta)/k)
      print(f"Somma a 1 : {abs(v.sum() - 1) < 1e-10}")
      residuo = np.abs(v - v_old).sum()
      print(f"Residuo : {residuo}")
      if residuo < 1e-8:
        print(f"Residuo : {residuo}")
        break
    return v, count_iterations, src_sub.nbytes+dst_sub.nbytes, bin_src_indexs_sub.sum()


if EDGES_RANGES_ANALYSIS:
  sizes = [500, 1000, 2000, 4000]
  benchmark_rows = []


  for size in sizes:
    start = time.time()
    v_dense, iters_dense, dense_matrix_bytes, total_edges_dense = dense_implementation(size, src_indexs, dst_indexs)

    dense_time = time.time() - start

    print(f"Tempo totale dense {(dense_time) / 60} min")
    print(f"Tempo per iterazione dense {((dense_time) / 60) / iters_dense} min")
    start = time.time()
    v_sparse, iters_sparse, sparse_matrix_bytes, total_edges_sparse = sparse_implementation(size, src_indexs, dst_indexs)

    sparse_time = time.time() - start
    print(f"Tempo totale sparse {(sparse_time) / 60} min")
    print(f"Tempo per iterazione sparse {((sparse_time) / 60) / iters_sparse} min")
    benchmark_rows.append({
    "nodes": size,
    "edges_dense": total_edges_dense,
    "edges_sparse": total_edges_sparse,
    "dense_time_per_iter_s": dense_time / iters_dense,
    "sparse_time_per_iter_s": sparse_time / iters_sparse,
    "dense_memory_mb": dense_matrix_bytes / 1024**2,
    "sparse_memory_mb": sparse_matrix_bytes / 1024**2,
    "max_abs_difference": np.abs(v_dense - v_sparse).max(),
    })

  df_benchmark = pd.DataFrame(benchmark_rows)
  df_benchmark


Iteration number 0
Somma a 1 : True
Residuo : 0.03965759999999964
Iteration number 1
Somma a 1 : True
Residuo : 0.00799076522666658
Iteration number 2
Somma a 1 : True
Residuo : 0.0003082257411837477
Iteration number 3
Somma a 1 : True
Residuo : 2.884940622968667e-05
Iteration number 4
Somma a 1 : True
Residuo : 1.7308352359812068e-06
Iteration number 5
Somma a 1 : True
Residuo : 9.027977561706924e-08
Iteration number 6
Somma a 1 : True
Residuo : 8.285809812141193e-09
Residuo : 8.285809812141193e-09
Tempo totale dense 0.00023810466130574543 min
Tempo per iterazione dense 3.401495161510649e-05 min
Iteration number 0
Somma a 1 : True
Residuo : 0.03965759999999964
Iteration number 1
Somma a 1 : True
Residuo : 0.007990765226666578
Iteration number 2
Somma a 1 : True
Residuo : 0.00030822574118374857
Iteration number 3
Somma a 1 : True
Residuo : 2.8849406229687537e-05
Iteration number 4
Somma a 1 : True
Residuo : 1.7308352359812068e-06
Iteration number 5
Somma a 1 : True
Residuo : 9.02797756

In [27]:
df_benchmark

,nodes,edges_dense,edges_sparse,dense_time_per_iter_s,sparse_time_per_iter_s,dense_memory_mb,sparse_memory_mb,max_abs_difference
0,500,28,28,0.000516,0.000384,1.907349,0.000427,0.000000e+00
1,1000,62,62,0.005133,0.000203,7.629395,0.000946,2.168404e-19
2,2000,324,324,0.002724,0.000843,30.517578,0.004944,1.734723e-18
3,4000,1288,1288,0.002962,0.002687,122.070312,0.019653,8.673617e-19


In [21]:
# Sparse implementation

import time

start = time.time()
v, count_iterations, sparse_bytes, _ =  sparse_implementation(n, src_indexs, dst_indexs)
end_time = time.time() - start
print(f"Tempo totale {(end_time) / 60} min")
print(f"Tempo per iterazione {((end_time) / 60) / count_iterations} min")
print(f"Spazio occupato mb {sparse_bytes / 1024**2}")

row_numpy_implementation = {
    "nodes": n,
    "total_time": (end_time) / 60,
    "time_per_iter_s": ((end_time) / 60) / count_iterations,
    "memory_mb": sparse_bytes / 1024**2,
    }

Iteration number 0
Somma a 1 : True
Residuo : 0.8929122204700866
Iteration number 1
Somma a 1 : True
Residuo : 0.25936221457527414
Iteration number 2
Somma a 1 : True
Residuo : 0.08258915022830149
Iteration number 3
Somma a 1 : True
Residuo : 0.029225403209928412
Iteration number 4
Somma a 1 : True
Residuo : 0.011283745996659867
Iteration number 5
Somma a 1 : True
Residuo : 0.004895587733061943
Iteration number 6
Somma a 1 : True
Residuo : 0.0023841084626879203
Iteration number 7
Somma a 1 : True
Residuo : 0.0013014631611363912
Iteration number 8
Somma a 1 : True
Residuo : 0.0007824141502705121
Iteration number 9
Somma a 1 : True
Residuo : 0.0005304947495465045
Iteration number 10
Somma a 1 : True
Residuo : 0.0004087387831894614
Iteration number 11
Somma a 1 : True
Residuo : 0.0003315114207037523
Iteration number 12
Somma a 1 : True
Residuo : 0.000273822874677421
Iteration number 13
Somma a 1 : True
Residuo : 0.00022788826317241608
Iteration number 14
Somma a 1 : True
Residuo : 0.00019

In [22]:
indexs_of_top20 = np.argsort(v)[::-1][:20]
in_degree = np.bincount(dst_indexs, minlength=n)

df_top20 = df_comments[["userID","userDisplayName"]].drop_duplicates()

df_userID_count_comments = df_comments[["userID", "commentID"]].groupby("userID").count().reset_index()

for idx in range(len(indexs_of_top20)):
  page_rank_value = v[indexs_of_top20[idx]]
  user_id = users_converted_indexes[indexs_of_top20[idx]]
  degree = in_degree[indexs_of_top20[idx]]
  mask = df_top20.userID == user_id
  df_top20.loc[mask, "page_rank"] = page_rank_value
  df_top20.loc[mask, "in_degree"] = degree

df_top20 = df_top20[df_top20.page_rank.notna()]
df_top20 = df_top20.sort_values(by="page_rank", ascending=False).drop_duplicates(subset="userID")

df_top20 = df_top20.merge(df_userID_count_comments, how="left", left_on="userID", right_on="userID")
df_top20 = df_top20.rename(columns={"commentID": "count_comments"})

df_top20["degreOnComments"] = df_top20["in_degree"] / df_top20["count_comments"] * 100


In [23]:
df_top20

,userID,userDisplayName,page_rank,in_degree,count_comments,degreOnComments
0,51878992,Socrates,0.001724,3549.0,4279,82.939939
1,78343265,KMW,0.001146,2967.0,2190,135.479452
2,68938663,Marge Keller,0.001064,2547.0,7049,36.132785
3,66332358,"Red Sox, ‘04, ‘07, ‘13, ‘18",0.000946,2153.0,1636,131.601467
4,2073520,ChristineMcM,0.000901,2146.0,2328,92.182131
5,73444633,AACNY,0.000832,2206.0,4891,45.103251
6,72040400,Roland Deschain,0.000804,1841.0,1310,140.534351
7,38331232,Bruce Rozenblit,0.000782,1813.0,959,189.051095
8,40118532,NM,0.000772,1801.0,2197,81.975421
9,5356803,Mon Ray,0.000748,1774.0,1702,104.230317


# Page Ranking Results discussion Kristof and Keller

In [75]:
idx_kristof = np.searchsorted(users_converted_indexes, 45616064)
idx_keller = np.searchsorted(users_converted_indexes, 68938663)

In [80]:
kristof_mask = dst_indexs == idx_kristof

src_kristof = src_indexs[kristof_mask]

print(f"Mean Page rank overall {v.mean()}")

print(f"Mean Page rank Kristof {v[src_kristof].mean()}")

print(f"Total number of top20 on comments of Kristof {len(set(src_kristof).intersection(set(indexs_of_top20)))}")

keller_mask = dst_indexs == idx_keller

src_keller = src_indexs[keller_mask]

print(f"Mean Page rank Keller {v[src_keller].mean()}")

print(f"Total number of top20 on comments of Keller {len(set(src_keller).intersection(set(indexs_of_top20)))}")




Mean Page rank overall 3.8141152778201556e-06
Mean Page rank Kristof 3.719723156021232e-05
Total number of top20 on comments of Kristof 6
Mean Page rank Keller 4.4465685093316985e-05
Total number of top20 on comments of Keller 12


# Spark Implementation

In [27]:
from pyspark.sql import SparkSession
import numpy as np
from operator import add
import time

spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext
k = sc.defaultParallelism

bin_src_indexs = np.bincount(src_indexs,  minlength=n)

n = len(users_converted_indexes)
Beta = 0.85

v_init = [(i, 1/n) for i in range(n)]
dangling_set = set(np.where(bin_src_indexs == 0)[0].tolist())
v_spark= sc.parallelize(v_init).partitionBy(k).cache()
sparse_matrix_spark = sc.parallelize(list(zip(src_indexs.tolist(), dst_indexs.tolist()))).groupByKey().mapValues(list).partitionBy(k).cache()
zeros_for_missing_nodes = sc.parallelize([(i, 0) for i in range(n)]).partitionBy(k).cache()

def calculate_j_elem(list_of_dst: list, v_value: float):
  return [(dst, v_value/len(list_of_dst)) for dst in list_of_dst]


start = time.time()
count_iterations = 0
for iter in range(50):
  print(f"Iteration number {iter}")
  count_iterations += 1
  danglings = v_spark.filter(lambda row: row[0] in dangling_set)
  S = danglings.values().sum()

  sparse_matrix_spark_joined = sparse_matrix_spark.join(v_spark)

  v_new = sparse_matrix_spark_joined.flatMap(lambda row: calculate_j_elem(list_of_dst=row[1][0], v_value=row[1][1])).reduceByKey(add).union(zeros_for_missing_nodes).reduceByKey(add).mapValues(lambda r: Beta*(r + S/n) + (1-Beta)/n)
  v_joined = v_spark.join(v_new)
  residuo = v_joined.map(lambda row: abs(row[1][0] - row[1][1])).reduce(add)
  print(f"Residuo : {residuo}")
  if residuo < 1e-8:
    break
  v_spark = v_new.cache()

end_time = time.time() - start
print(f"Tempo totale {(end_time) / 60} min")
print(f"Tempo per iterazione {((end_time) / 60) / count_iterations} min")

row_spark_implementation = {
    "nodes": n,
    "total_time": (end_time) / 60,
    "time_per_iter_s": ((end_time) / 60) / count_iterations,
    "memory_mb": "",
    }



Iteration number 0
Residuo : 0.8929122204711433
Iteration number 1
Residuo : 0.25936221457484754
Iteration number 2
Residuo : 0.08258915022834118
Iteration number 3
Residuo : 0.029225403209925765
Iteration number 4
Residuo : 0.011283745996639841
Iteration number 5
Residuo : 0.004895587733075947
Iteration number 6
Residuo : 0.002384108462693501
Iteration number 7
Residuo : 0.0013014631611358435
Iteration number 8
Residuo : 0.0007824141502688537
Iteration number 9
Residuo : 0.0005304947495462634
Iteration number 10
Residuo : 0.0004087387831899232
Iteration number 11
Residuo : 0.00033151142070446296
Iteration number 12
Residuo : 0.0002738228746782033
Iteration number 13
Residuo : 0.0002278882631722439
Iteration number 14
Residuo : 0.00019097322792258493
Iteration number 15
Residuo : 0.00016051688249263098
Iteration number 16
Residuo : 0.00013575328522002117
Iteration number 17
Residuo : 0.00011497674673122084
Iteration number 18
Residuo : 9.74324691955731e-05
Iteration number 19
Residuo :

In [28]:
# unwrap
RDD_v = v_spark.takeOrdered(20, key=lambda x: -x[1])

In [59]:
in_degree = np.bincount(dst_indexs, minlength=n)

df_top20_spark = df_comments[["userID","userDisplayName"]].drop_duplicates()

df_userID_count_comments_spark = df_comments[["userID", "commentID"]].groupby("userID").count().reset_index()

for idx in range(len(RDD_v)):
  node_idx = RDD_v[idx][0]
  page_rank_value = RDD_v[idx][1]
  user_id = users_converted_indexes[node_idx]
  degree = in_degree[node_idx]
  mask = df_top20_spark.userID == user_id
  df_top20_spark.loc[mask, "page_rank"] = page_rank_value
  df_top20_spark.loc[mask, "in_degree"] = degree

df_top20_spark = df_top20_spark[df_top20_spark.page_rank.notna()]
df_top20_spark = df_top20_spark.sort_values(by="page_rank", ascending=False).drop_duplicates(subset="userID")

df_top20_spark = df_top20_spark.merge(df_userID_count_comments_spark, how="left", left_on="userID", right_on="userID")
df_top20_spark = df_top20_spark.rename(columns={"commentID": "count_comments"})

df_top20_spark["degreOnComments"] = df_top20_spark["in_degree"] / df_top20_spark["count_comments"] * 100


# Benchmark Spark vs Numpy

In [64]:
df_joined = df_top20.merge(df_top20_spark, how="left", left_on="userID", right_on="userID", suffixes=("_dense", "_spark"))

df_joined["diffence_sparse_dense"] = df_joined["page_rank_dense"] - df_joined["page_rank_spark"]
df_joined["diffence_degreOnComments_sparse_dense"] = df_joined["degreOnComments_dense"] - df_joined["degreOnComments_spark"]

df_joined[["userID", "userDisplayName_dense", "diffence_sparse_dense", "diffence_degreOnComments_sparse_dense"]]

,userID,userDisplayName_dense,diffence_sparse_dense,diffence_degreOnComments_sparse_dense
0,51878992,Socrates,-0.00,0.00
1,78343265,KMW,-0.00,0.00
2,68938663,Marge Keller,-0.00,0.00
3,66332358,"Red Sox, ‘04, ‘07, ‘13, ‘18",-0.00,0.00
4,2073520,ChristineMcM,-0.00,0.00
5,73444633,AACNY,-0.00,0.00
6,72040400,Roland Deschain,-0.00,0.00
7,38331232,Bruce Rozenblit,0.00,0.00
8,40118532,NM,-0.00,0.00
9,5356803,Mon Ray,0.00,0.00


In [66]:
# max difference
df_joined["diffence_sparse_dense"].abs().max()

3.903127820947816e-18

In [32]:
df_benchmark_numpy_vs_spark = pd.DataFrame([row_numpy_implementation, row_spark_implementation])
df_benchmark_numpy_vs_spark

,nodes,total_time,time_per_iter_s,memory_mb
0,262184,2.540290,0.050806,30.018433
1,262184,9.948269,0.198965,
